In [ ]:
import numpy as np
import hcipy as hp
import hcipy as hcipy
from hcipy import *
import matplotlib.pyplot as plt
from utils.magaox import make_magaox_bump_mask
from utils.apodized_mask_generator import make_magaox_piaa_lyot_stop
from utils.PIAACMC import prep_map, measure_shift_integer
import csv
import scipy.ndimage as ndi

datadir = '/media/armand/EXTRA/Sync/Master-Leiden/Y1/First Project/PIAACMC/magaox_data'


Define constants, 

INIT pupil and aperture for Magellan, may change to MagAO-X

Supersample the pupil by factor 8 for more detail

In [ ]:
# Telescope / instrument constants
aperture_diameter = 6.5            # m
effective_focal_length = 71.5   # m

pupil_grid = hcipy.make_pupil_grid(500, diameter=aperture_diameter)

magellan_pupil_generator = make_magaox_bump_mask()
telescope_pupil = hp.evaluate_supersampled(magellan_pupil_generator, pupil_grid, 8)

hcipy.imshow_field(telescope_pupil, cmap='gray')
plt.colorbar()
plt.xlabel('x [m]')
plt.ylabel('y [m]')
plt.show()


Create and select the filters: 

- Narrowband 875 filter $\lambda$ = 875nm, bandwidth = 3\%
- Broadband z' filter (here z): $\lambda$ = 908nm, bandwidth = 14\%

In [ ]:
# Define and select filter
filter_875 = {'name': '875 nm', 'wavelength': 875e-9, 'bandwidth': 0.03} 
filter_z   = {'name': 'z band', 'wavelength': 908e-9, 'bandwidth': 0.13/0.908}
# 875 filter is to narrowband, not enough flux for the detection of faint objects. Use z band instead.
# filter = filter_875
filter = filter_z
wavelength = filter['wavelength']
bandwidth = filter['bandwidth']


Propagate a Monochomatic (at central wavelength of the filter) wavefront towards the focal plane.\
Propagate a bandwidth for each filter

In [ ]:
wavelength = filter['wavelength']
bandwidth = filter['bandwidth']
wavefront = hcipy.Wavefront(telescope_pupil, wavelength=wavelength) # create wavefront object

focal_grid = hcipy.make_focal_grid(q=8, num_airy=16, pupil_diameter=aperture_diameter,
                                focal_length=effective_focal_length, reference_wavelength=wavelength)

fraun = hcipy.FraunhoferPropagator(pupil_grid, focal_grid, focal_length=effective_focal_length)

focal_image = fraun.forward(wavefront) #plot the monochromatic PSF

focal_total = 0
for wlen in wavelength * np.linspace(1 - bandwidth / 2., 1 + bandwidth / 2., 11):
    wavefront = hcipy.Wavefront(telescope_pupil, wlen)
    focal_total += fraun.forward(wavefront).power

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].set_title('Pupil Plane')
ax[1].set_title('Focal Plane')
ax[2].set_title('Focal Plane (Broadband)')

hcipy.imshow_field(wavefront.amplitude, grid_units = aperture_diameter, ax=ax[0], cmap='inferno')
ax[0].set_xlabel(r'Pupil plane distance [$m$]')
ax[0].set_ylabel(r'Pupil plane distance [$m$]')
ax[0].set_title(filter['name'] + ' — pupil plane')
ax[0].set_aspect('equal')
plt.colorbar(ax[0].images[0], ax=ax[0], label='Amplitude')

hcipy.imshow_field(np.log10(focal_image.power / focal_image.power.max()), grid_units=1e-6, ax=ax[1],cmap='inferno')
ax[1].set_xlabel(r'Focal plane distance [$\mu m$]')
ax[1].set_ylabel(r'Focal plane distance [$\mu m$]')
ax[1].set_title(filter['name'] + ' — monochromatic PSF')
ax[1].set_aspect('equal')
plt.colorbar(ax[1].images[0], ax=ax[1], label='Log10(power)')

hcipy.imshow_field(np.log10(focal_total / focal_total.max()), grid_units=1e-6, ax=ax[2],cmap='inferno')
ax[2].set_xlabel(r'Focal plane distance [$\mu m$]')
ax[2].set_ylabel(r'Focal plane distance [$\mu m$]')
ax[2].set_title(filter['name'] + ' — broadband PSF')
ax[2].set_aspect('equal')

plt.colorbar(ax[2].images[0], ax=ax[2], label='Log10(power)')
plt.show()


Import CSV for PIAA and FPM

In [ ]:
piaa0_csv = np.loadtxt('utils/piaa0_CaF2.csv', delimiter=',', skiprows=1)   # [radius_m, cut_depth_m]
piaa1_csv = np.loadtxt('utils/piaa1_CaF2.csv', delimiter=',', skiprows=1)
radial_d0, profile0 = piaa0_csv[:, 0], -piaa0_csv[:, 1]
radial_d1, profile1 = piaa1_csv[:, 0], -piaa1_csv[:, 1]
min_0=np.min(radial_d0)
min_1=np.min(radial_d1)
profile0=profile0-min_0
profile1=profile1-min_1

def load_fpm_csv(path):
    """Load the Keyence VK-X1000 height map (FPM.csv). Returns (height_um, pixel_size_m)."""
    with open(path) as f:
        rows = list(csv.reader(f))
    header_i = next(i for i, row in enumerate(rows) if row and row[0] == 'Height')
    meta = {row[0]: row[1] for row in rows[:header_i] if len(row) >= 2}
    pixel_size = float(meta['XY calibration']) * 1e-6          # m/px
    height_um = np.array([[float(v) for v in row] for row in rows[header_i + 1:] if row])
    height_um[height_um < 0.5] = 0                             # zero the noise floor
    return height_um, pixel_size

def find_fpm_center(height_um):
    """Ring centre = centre of the central disc (ring 0, height ~1.26 um).

    The innermost ring is a filled disc of uniform height; its centroid is the
    geometric centre of the whole ring pattern. Isolating it by height ignores
    the outer rings and outer values, which can bias the full-disc center.
    """
    central_disc = (height_um > 1.1) & (height_um < 1.5)
    labels, n_components = ndi.label(central_disc)
    sizes = ndi.sum(central_disc, labels, range(1, n_components + 1))
    largest = np.argmax(sizes) + 1
    center_y, center_x = ndi.center_of_mass(labels == largest)
    return center_x, center_y

fpm_height, fpm_pixel_size = load_fpm_csv('utils/FPM.csv')
fpm_center = find_fpm_center(fpm_height)
print('FPM centre [px]:', fpm_center, '  pixel size [m]:', fpm_pixel_size)


Plot lense profile and FPM shape

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].plot(radial_d0 * 1e3, profile0, label='PIAA0', color='purple')
ax[0].plot(radial_d1 * 1e3, profile1, label='PIAA1', color='hotpink')
ax[0].set_xlabel('Radius [mm]')
ax[0].set_ylabel('height [µm]')
ax[0].set_title('Measured PIAA lens surfaces')
ax[0].legend()

ax[1].imshow(fpm_height, cmap='inferno')
ax[1].scatter(*fpm_center, c='white', s=50, marker='x', label=f'Measured centre ({fpm_center[0]:.1f}, {fpm_center[1]:.1f})')
# ax[1].set_xlim(330, 700)
# ax[1].set_ylim(150, 520)
ax[1].set_xlabel('Pixel index')
ax[1].set_ylabel('Height [µm]')
ax[1].set_title('Measured FPM height map')
ax[1].legend()
plt.colorbar(ax[1].images[0], ax=ax[1], label='Height [µm]')
plt.show()


Create the lense propagators 

In [ ]:
d_pupil_piaa0 = 37.2e-3   # m, pupil plane -> PIAA0
d_piaa0_piaa1 = 60.2e-3   # m, PIAA0 -> PIAA1
pupil_diameter = 8.6e-3   # m, pupil diameter (for the wavefront)
ref_ind_lenses = hcipy.get_refractive_index('CAF2')   # n(lambda)

piaa_pupil_grid = hcipy.make_pupil_grid(500, diameter=1.3*pupil_diameter)
aperture = hp.evaluate_supersampled(make_magaox_bump_mask(aperture_scale="pupil"), piaa_pupil_grid, 8)

prop_pupil_piaa0 = hp.FresnelPropagator(aperture.grid,d_pupil_piaa0) #AngularSpectrumPropagator
prop_piaa0_piaa1 = hp.FresnelPropagator(aperture.grid,d_piaa0_piaa1)
prop_piaa1_pupil = hp.FresnelPropagator(aperture.grid,d_piaa0_piaa1+d_pupil_piaa0)

prop_LS_piaa1 = hp.FresnelPropagator(aperture.grid,d_pupil_piaa0)
prop_piaa1_piaa0 = hp.FresnelPropagator(aperture.grid,d_piaa0_piaa1)
prop_piaa0_LS = hp.FresnelPropagator(aperture.grid,d_piaa0_piaa1+d_pupil_piaa0)


sag0 = hp.Field(np.interp(aperture.grid.as_('polar').r,radial_d0, profile0),aperture.grid)
sag1 = hp.Field(np.interp(aperture.grid.as_('polar').r,radial_d1, profile1),aperture.grid)

piaa0 = hp.SurfaceApodizer(sag0, ref_ind_lenses)
piaa1 = hp.SurfaceApodizer(sag1, ref_ind_lenses)

separation = 0
wf_pupil = hp.Wavefront(aperture*np.exp(2j * np.pi * separation * aperture.grid.x / pupil_diameter))

wf_pupil.wavelength = wavelength
wf_pupil.total_power = 1.0
wf_piaa0 = piaa0.forward(prop_pupil_piaa0.forward(wf_pupil))
wf_piaa1 = piaa1.forward(prop_piaa0_piaa1.forward(wf_piaa0))
wf_apodized = prop_piaa1_pupil.backward(wf_piaa1)


#plot lenses
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].set_title('PIAA0 sag [m]')
ax[1].set_title('PIAA1 sag [m]')
ax[2].set_title('Apodized pupil power')
hcipy.imshow_field(piaa0.surface_sag, cmap='gray', grid_units=1e-3, ax=ax[0])
hcipy.imshow_field(piaa1.surface_sag, cmap='gray', grid_units=1e-3, ax=ax[1])
hcipy.imshow_field(wf_apodized.power, cmap='gray', grid_units=1e-3, ax=ax[2])
ax[0].set_xlabel('x [m]')
ax[0].set_ylabel('y [m]')
ax[1].set_xlabel('x [m]')
ax[1].set_ylabel('y [m]')
ax[2].set_xlabel('x [m]')
ax[2].set_ylabel('y [m]')
plt.colorbar(ax[0].images[0], ax=ax[0], label='Sag [m]')
plt.colorbar(ax[1].images[0], ax=ax[1], label='Sag [m]')
plt.colorbar(ax[2].images[0], ax=ax[2], label='Power')
plt.show()


In [ ]:
# Focal-plane scale: 1 lambda/D in the focal plane (in metres)
F_number = 69
spatial_resolution = wavelength * F_number            # 1 lambda/D (~62.6 µm)
fpm_radius = 0.75 * spatial_resolution                # FPM radius = 0.75 lambda/D (~47 µm, papers 2 & 3)
print(f'Focal-plane scale (1 lambda/D) = {spatial_resolution * 1e6:.2f} µm,  FPM radius = {fpm_radius * 1e6:.2f} µm')

# Focal grid that covers the FPM, and the propagator that reaches it
x_diam = fpm_height.shape[1]
y_diam = fpm_height.shape[0]

cor_grid = hp.make_focal_grid(q=200, num_airy=5, spatial_resolution=spatial_resolution)
# cor_grid = make_pupil_grid([x_diam,y_diam],[x_diam*fpm_pixel_size,y_diam*fpm_pixel_size])

n_mask = hp.get_refractive_index('SILICA')            # FPM substrate = fused silica
prop_piaa1_focal = hp.FraunhoferPropagator(piaa_pupil_grid, cor_grid, focal_length=F_number * pupil_diameter)

# Focal-plane image right before the FPM (apodized pupil, no mask yet)
wavefront_focal = prop_piaa1_focal.forward(wf_apodized)

plt.figure(figsize=(6, 5))
hcipy.imshow_field(np.log10(wavefront_focal.power / wavefront_focal.power.max()), grid_units=1e-6, cmap='inferno')
plt.xlabel(r'Focal plane distance [$\mu m$]')
plt.ylabel(r'Focal plane distance [$\mu m$]')
plt.title('Focal plane after PIAA lenses (before FPM)')
plt.colorbar(label='Log10(power)')
plt.show()


In [ ]:
n_rings=10
wl=0.908e-6
F_number=69
SR=wl*F_number
expected_max_radius_norm=0.75
expected_heights=np.array([1.26196831, 0.94074998, 0.77836148, 0.6366583 , 0.7739299 , 3.19782967, 0.85539274, 0.79005693, 0.88346143, 0.87813155]) #microns

# --- FPM ring design: the attributes the coworker's class would have held ---
# CONFIRM: radii (evenly spaced to 0.75 lambda/D) and n_mask (FPM material)
radii = np.linspace(expected_max_radius_norm / n_rings, expected_max_radius_norm, n_rings) * SR  # metres
# cor_grid = hp.make_focal_grid(q=100, num_airy=2, spatial_resolution=SR) 
n_mask = hp.get_refractive_index('SILICA')

def make_rings(radii, cor_grid, supersample_factor):
        '''

        '''

        masks_in=dict()
        masks_out=dict()

        for i in range(len(radii)-1,-1,-1):
            if i==0:
                ca1=make_circular_aperture(radii[i]*2)
                supersampled_ca1=evaluate_supersampled(ca1,cor_grid,supersample_factor)
                masks_in[i]=1-supersampled_ca1
            else:
                ca1=make_circular_aperture(radii[i]*2)
                ca2=make_circular_aperture(radii[i-1]*2)
                supersampled_ca1=evaluate_supersampled(ca1,cor_grid,supersample_factor)
                supersampled_ca2=evaluate_supersampled(ca2,cor_grid,supersample_factor)
                masks_in[i]=1-supersampled_ca1+supersampled_ca2
            masks_out[i]=1-masks_in[i]

        return masks_out

def make_FPM_w_rings(radii, cor_grid, supersample_factor, n_mask, thicknesses):
    '''
    Input thicknesses needs to be given in MICRONS.
    Returns FPM apodizer, where the surface sag is in METERS.
    '''

    rings=make_rings(radii, cor_grid, supersample_factor)

    for i in range(len(radii)):
        if i==0:
            FPM=rings[0]*thicknesses[0]*1e-6
        else:
            FPM+=rings[i]*thicknesses[i]*1e-6

    FPM_apodizer=SurfaceApodizer(FPM,n_mask)

    return FPM_apodizer

supersample_factor = 8
FPM_apodizer = make_FPM_w_rings(radii, cor_grid, supersample_factor, n_mask, expected_heights)

# apply the FPM to the apodized pupil at the F#=69 focal plane (gives the cell below its wf_fpm)
prop_piaa1_focal = hp.FraunhoferPropagator(piaa_pupil_grid, cor_grid, focal_length=F_number * pupil_diameter)
wf_fpm = FPM_apodizer.forward(prop_piaa1_focal.forward(wf_apodized))

# quick check of the FPM surface
hp.imshow_field(FPM_apodizer.surface_sag * 1e6, cmap='inferno', grid_units=1e-6)
plt.colorbar(label='FPM surface height [um]')
plt.title('FPM surface (from rings)')
plt.show()



Implement FPM using the csv data


In [ ]:
def make_measured_fpm(height_um, pixel_size, center, cor_grid, refractive_index):
    """Sample the measured FPM height map onto cor_grid and return its SurfaceApodizer.

    height_um        : 2D measured height map, in microns
    pixel_size       : image pixel size, in metres
    center           : (center_x, center_y) of the rings, in pixels
    cor_grid         : focal grid the FPM is defined on
    refractive_index : FPM substrate refractive index
    """
    center_x, center_y = center
    grid_x = cor_grid.x.reshape(cor_grid.dims)          # [m]
    grid_y = cor_grid.y.reshape(cor_grid.dims)          # [m]
    column = center_x + grid_x / pixel_size
    row    = center_y - grid_y / pixel_size             # minus: row 0 = top = +y
    sag = ndi.map_coordinates(height_um, [row, column], order=1, mode='constant', cval=0.0) * 1e-6   # [m]
    return hp.SurfaceApodizer(hp.Field(sag.ravel(), cor_grid), refractive_index)

fpm_apodizer = make_measured_fpm(fpm_height, fpm_pixel_size, fpm_center, cor_grid, n_mask)
wavefront_fpm = fpm_apodizer.forward(prop_piaa1_focal.forward(wf_apodized))

# quick check of the FPM surface
hp.imshow_field(fpm_apodizer.surface_sag * 1e6, cmap='inferno', grid_units=1e-6)
plt.colorbar(label='FPM surface height [µm]')
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.title('FPM surface (measured)')
plt.show()


Propagate through the Lyot stop and inverse PIAA lenses to the final focal plane

In [ ]:
# FPM plane -> Lyot pupil plane
prop_fpm_lyot = hp.FraunhoferPropagator(cor_grid, piaa_pupil_grid, focal_length=F_number * pupil_diameter)
wavefront_lyot = prop_fpm_lyot.forward(wavefront_fpm)

# Lyot stop: blocks the light diffracted outside the apodized pupil by the FPM
lyot_stop = hp.evaluate_supersampled(make_magaox_piaa_lyot_stop(), piaa_pupil_grid, 8)
wavefront_lyot_stopped = hp.Wavefront(wavefront_lyot.electric_field * lyot_stop, wavelength=wavelength)

# Inverse PIAA lenses back to the detector pupil
wavefront_pupil_out = prop_piaa0_LS.forward(piaa0.backward(
                        prop_piaa1_piaa0.forward(piaa1.backward(
                        prop_LS_piaa1.forward(wavefront_lyot_stopped)))))

# Detector focal plane: the final coronagraphic PSF
detector_grid = hp.make_focal_grid(q=4, num_airy=16, pupil_diameter=pupil_diameter,
                                   focal_length=F_number * pupil_diameter, reference_wavelength=wavelength)
prop_detector = hp.FraunhoferPropagator(aperture.grid, detector_grid, focal_length=F_number * pupil_diameter)
psf = prop_detector.forward(wavefront_pupil_out)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
hcipy.imshow_field(np.log10(wavefront_lyot.power / wavefront_lyot.power.max()), cmap='inferno', ax=ax[0])
ax[0].set_title('Lyot pupil (after FPM)')
ax[0].set_aspect('equal')
hcipy.imshow_field(np.log10(psf.power / psf.power.max()), grid_units=1e-6, cmap='inferno', ax=ax[1])
ax[1].set_title('Final coronagraphic PSF')
ax[1].set_xlabel(r'Detector plane [$\mu m$]')
ax[1].set_ylabel(r'Detector plane [$\mu m$]')
plt.show()
